# Lomas: Dummy GS

## Step 1: Install the library
To interact with the secure server on which the data is stored, Dr.Antartica first needs to install the library `lomas-client` on her local developping environment. 

It can be installed via the pip command:

In [ ]:
# !pip install lomas-client

In [ ]:
%load_ext autoreload
%autoreload 2

from lomas_client import Client
import numpy as np

In [ ]:
import pyarrow

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator (queen Icebergina) and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

She will only be able to query on the real dataset if the queen Icebergina has previously made her an account in the database, given her access to the PENGUIN dataset and has given her some epsilon and delta credit.

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "http://localhost:48080"
# os.environ["LOMAS_CLIENT_KEYCLOAK_URL"] = "http://localhost:4442"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# # We set these ones because they are specific to this notebook.

USER_NAME = "Pauline"
os.environ["LOMAS_CLIENT_CLIENT_ID"] = USER_NAME
os.environ["LOMAS_CLIENT_CLIENT_SECRET"] = "in6tx61l2wLNc36GlwviCzujHyuOgD5Y"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "DUMMY_GS"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(client_id = "Dr.Antartica") takes precedence over setting the "LOMAS_CLIENT_CLIENT_ID"
# environment variable.

In [ ]:
client = Client()

## GS queries

In [ ]:
import polars as pl
import opendp.prelude as dp

In [ ]:
NB_ROWS = 100
SEED=42

In [ ]:
dummy_lf = client.get_dummy_dataset(nb_rows=NB_ROWS, seed=SEED, lazy=True)

In [ ]:
lower = client.get_dataset_metadata()["columns"]["nb_hopitalizations_in_year"]["lower"]
upper = client.get_dataset_metadata()["columns"]["nb_hopitalizations_in_year"]["upper"]

In [ ]:
avg_hosp_polars = (
    dummy_lf
    .group_by(['age_group', 'sex', 'after_diagnosis'])
    .agg(
        pl.col("nb_hopitalizations_in_year").dp.mean(bounds=(lower, upper), scale=(100.0,1))
    )
)

In [ ]:
res = client.opendp.query(avg_hosp_polars, dummy = False, nb_rows = NB_ROWS, seed=SEED)
res.result.value.sort(['age_group', 'sex', 'after_diagnosis'])

### query 2

In [ ]:
lower_bound_los = client.get_dataset_metadata()["columns"]["length_of_stay"]["lower"]
upper_bound_los = client.get_dataset_metadata()["columns"]["length_of_stay"]["upper"]

In [ ]:
avg_los_polars = (
    dummy_lf
    .group_by(['age_group', 'sex', 'after_diagnosis'])
    .agg(
        pl.col('length_of_stay').dp.mean(bounds=(lower_bound_los, upper_bound_los), scale = (100, 0.1)).alias('avg_length_of_stay')
    )
)

In [ ]:
res = client.opendp.query(avg_los_polars, dummy = False, nb_rows = NB_ROWS, seed=SEED)
res.result.value.sort(['age_group', 'sex', 'after_diagnosis']).sort(['age_group', 'sex', 'after_diagnosis'])

### Query 3

In [ ]:
columns = dummy_lf.select(pl.col("^.*diagnostic.*$")).collect().columns

In [ ]:
# Loop on each column (Query Lomas)
dfs = []
for column in columns:
    plan = (dummy_lf
        .group_by([column])
        .agg(
            dp.len(scale=1)
        )
    )
    res = client.opendp.query(plan, dummy = False, nb_rows = NB_ROWS, seed=SEED)
    df = res.result.value
    df = df.rename({ "len": f"len_{column}" })

    dfs.append(df)
final_df = pl.concat(dfs, how="horizontal")

In [ ]:
# Postprocess
diagnosis_count_pairs = [
    ("main_diagnostic", "len_main_diagnostic"),
    ("main_diagnostic_complement", "len_main_diagnostic_complement"),
] + [
    (f"supplementary_diagnostics_{i}", f"len_supplementary_diagnostics_{i}")
    for i in range(19)
]

long_format = pl.DataFrame()

for diag_col, count_col in diagnosis_count_pairs:
    temp = final_df.select([
        pl.col(diag_col).alias("diagnosis"),
        pl.col(count_col).alias("count")
    ])
    long_format = pl.concat([long_format, temp], how="vertical")

result = (
    long_format
    .filter(pl.col("diagnosis") != "") 
    .group_by("diagnosis")
    .agg(pl.sum("count").alias("total_count"))
    .sort("total_count", descending=True)
)

print(result.head(10))
